In [ ]:
from huggingface_hub import login
login("token")
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from unsloth import FastLanguageModel
import torch
import os
import json

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
ds = load_dataset("minnesotanlp/scholawrite")
test_df = pd.DataFrame(ds["test_small"])
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
# model_name = "minnesotanlp/scholawrite-llama3.1-8b-writing"
okenizer = FastLanguageModel.from_pretrained(
  model_name=model_name,
  max_seq_length=4096,
  load_in_4bit=True,
  dtype=None,
)
FastLanguageModel.for_inference(model)


README.md:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/50.5M [00:00<?, ?B/s]

data/test_small-00000-of-00001.parquet:   0%|          | 0.00/13.5M [00:00<?, ?B/s]

data/all_sorted-00000-of-00002.parquet:   0%|          | 0.00/23.3M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [ ]:
for proj_idx in range(1,6,1):
    project_test = test_df[test_df['project'] == proj_idx].sort_values(by="timestamp").reset_index(drop=True)
    project_test_dict = project_test.to_dict(orient="records")
    n = len(project_test_dict)
    file_path = f"project_{proj_idx}_llama_8b_zero.jsonl"
    with open(file_path, "a", encoding="utf-8") as f:
        for i in range(101):
            print(f"project {proj_idx} | sample {i}/{n}")
            before_text = project_test_dict[i]['before text']
            text = text_gen_prompt(before_text, project_test_dict[i]['label'])
            input_ids = tokenizer.apply_chat_template(text, max_length=4096, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    input_ids,
                    max_new_tokens=len(before_text) + 100,
                    do_sample=False,
                )

            response = tokenizer.batch_decode(outputs)
            response = response[0].split("<|start_header_id|>assistant<|end_header_id|>")[1].strip()
            response = response.replace("<|eot_id|>", "")
            response_clean = clean_text(response)
            project_test_dict[i]['index'] = i
            project_test_dict[i]['llama-8b-sw output'] = response_clean
            f.write(json.dumps(project_test_dict[i], ensure_ascii=False) + "\n")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


project 4 | sample 0/468
project 4 | sample 1/468
project 4 | sample 2/468
project 4 | sample 3/468
project 4 | sample 4/468
project 4 | sample 5/468
project 4 | sample 6/468
project 4 | sample 7/468
project 4 | sample 8/468
project 4 | sample 9/468
project 4 | sample 10/468
project 4 | sample 11/468
project 4 | sample 12/468
project 4 | sample 13/468
project 4 | sample 14/468
project 4 | sample 15/468
project 4 | sample 16/468
project 4 | sample 17/468
project 4 | sample 18/468
project 4 | sample 19/468
project 4 | sample 20/468
project 4 | sample 21/468
project 4 | sample 22/468
project 4 | sample 23/468
project 4 | sample 24/468
project 4 | sample 25/468
project 4 | sample 26/468
project 4 | sample 27/468
project 4 | sample 28/468
project 4 | sample 29/468
project 4 | sample 30/468
project 4 | sample 31/468
project 4 | sample 32/468
project 4 | sample 33/468
project 4 | sample 34/468
project 4 | sample 35/468
project 4 | sample 36/468
project 4 | sample 37/468
project 4 | sample 38/

In [ ]:
persona_definition = {
  "Idea Generation": "formulate and record initial thoughts and concepts.",
  "Idea Organization": "select the most useful materials and demarcate those generated ideas in a visually formatted way.",
  "Section Planning": "initially create sections and sub-level structures.",
  "Text Production": "translate your ideas into full languages, either from your language or borrowed sentences from an external source.",
  "Object Insertion": "insert visual claims of your arguments (e.g., figures, tables, equations, footnotes, itemized lists, etc.).",
  "Cross-reference": "link different sections, figures, tables, or other elements within a document through referencing commands.",
  "Citation Integration": "incorporate bibliographic references into a document and systematically link these references using citation commands.",
  "Macro Insertion": "incorporate predefined commands orc packages into a LaTeX document to alter its formatting.",
  "Fluency": "fix grammatical or syntactic errors in the text or LaTeX commands.",
  "Coherence": "logically link (1) any of the two or multiple sentences within the same paragraph; (2) any two subsequent paragraphs; or (3) objects to be consistent as a whole.",
  "Structural": "improve the flow of information by modifying the location of texts and objects.",
  "Clarity": "improve the semantic relationships between texts to be more straightforward and concise.",
  "Linguistic Style": "modify texts with your writing preferences regarding styles and word choices, etc.",
  "Scientific Accuracy": "update or correct scientific evidence (e.g., numbers, equations) for more accurate claims.",
  "Visual Formatting": "modify the stylistic formatting of texts, objects, and citations."
}


def text_gen_prompt(before_text, writing_intention):

    user_prompt = f"""You are a computer science researcher with extensive experience in scholarly writing. Here, you are writing a research paper in natural language processing using LaTeX.

You currently want to {persona_definition[writing_intention]}

Below is the paper you have written so far. Given the paper information below and the corresponding scholarly writing intention, please revise or add to the text to fulfill this writing intention.

You may insert, delete, or revise text at appropriate places in the given paper.

Please provide a complete output. Do not generate text that is nonsensical or unrelated to the given paper information.

{before_text}"""

    return [
        {"role": "user", "content": user_prompt}
    ]


def class_prompt(before_text):
    usr_prompt= f"""Here are all the possible writing intention labels:

Idea Generation: Formulate and record initial thoughts and concepts.
Idea Organization: Select the most useful materials and demarcate those generated ideas in a visually formatted way.
Section Planning: Initially create sections and sub-level structures.
Text Production: Translate their ideas into full languages, either from the writers' language or borrowed sentences from an external source.
Object Insertion: Insert visual claims of their arguments (e.g., figures, tables, equations, footnotes, itemized lists, etc.).
Cross-reference: Link different sections, figures, tables, or other elements within a document through referencing commands.
Citation Integration: Incorporate bibliographic references into a document and systematically link these references using citation commands.
Macro Insertion: Incorporate predefined commands or packages into a LaTeX document to alter its formatting.
Fluency: Fix grammatical or syntactic errors in the text or LaTeX commands.
Coherence: Logically link (1) any of the two or multiple sentences within the same paragraph; (2) any two subsequent paragraphs; or (3) objects to be consistent as a whole.
Structural: Improve the flow of information by modifying the location of texts and objects.
Clarity: Improve the semantic relationships between texts to be more straightforward and concise.
Linguistic Style: Modify texts with the writer's writing preferences regarding styles and word choices, etc.
Scientific Accuracy: Update or correct scientific evidence (e.g., numbers, equations) for more accurate claims.
Visual Formatting: Modify the stylistic formatting of texts, objects, and citations.

Identify the most likely next writing intention of a graduate researcher when writing the following LaTex paper draft. Your output should only be a label from the list above.

{before_text}"""

    return [
        {"role": "user", "content": usr_prompt}
    ]


#-----------------------------------------------------------------break for training prompt--------------------------------------------------------------------#


def class_prompt_train(before_text, intention_y):
    usr_prompt= f"""Here are all the possible writing intention labels:

Idea Generation: Formulate and record initial thoughts and concepts.
Idea Organization: Select the most useful materials and demarcate those generated ideas in a visually formatted way.
Section Planning: Initially create sections and sub-level structures.
Text Production: Translate their ideas into full languages, either from the writers' language or borrowed sentences from an external source.
Object Insertion: Insert visual claims of their arguments (e.g., figures, tables, equations, footnotes, itemized lists, etc.).
Cross-reference: Link different sections, figures, tables, or other elements within a document through referencing commands.
Citation Integration: Incorporate bibliographic references into a document and systematically link these references using citation commands.
Macro Insertion: Incorporate predefined commands or packages into a LaTeX document to alter its formatting.
Fluency: Fix grammatical or syntactic errors in the text or LaTeX commands.
Coherence: Logically link (1) any of the two or multiple sentences within the same paragraph; (2) any two subsequent paragraphs; or (3) objects to be consistent as a whole.
Structural: Improve the flow of information by modifying the location of texts and objects.
Clarity: Improve the semantic relationships between texts to be more straightforward and concise.
Linguistic Style: Modify texts with the writer's writing preferences regarding styles and word choices, etc.
Scientific Accuracy: Update or correct scientific evidence (e.g., numbers, equations) for more accurate claims.
Visual Formatting: Modify the stylistic formatting of texts, objects, and citations.

Identify the most likely next writing intention of a graduate researcher when editing the following LaTex paper draft. Your output should only be a label from the list above.

{before_text}"""

    return [
        {"role": "user", "content": usr_prompt},
        {"role": "assistant", "content": intention_y}
    ]

def text_gen_prompt_train(before_text, writing_intention, after_text):

    user_prompt = f"""You are a computer science researcher with extensive experience in scholarly writing. Here, you are writing a research paper in natural language processing using LaTeX.

You currently want to {persona_definition[writing_intention]}

Below is the paper you have written so far. Given the paper information below and the corresponding scholarly writing intention, please revise or add to the text to fulfill this writing intention.

You may insert, delete, or revise text at appropriate places in the given paper.

Please provide a complete output. Do not generate text that is nonsensical or unrelated to the given paper information.

{before_text}"""

    return [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": after_text}
    ]
def clean_text(text):
  text = re.sub(r"<del>.*?<\/del>", "", text, flags=re.DOTALL)

  text = re.sub(r"<same>(.*?)<\/same>", r"\1", text, flags=re.DOTALL)

  text = re.sub(r"<add>(.*?)<\/add>", r"\1", text, flags=re.DOTALL)

  tags_to_remove = ["<add>", "</add>", "<del>", "</del>", "<same>", "</same>"]
  for tag in tags_to_remove:
      text = text.replace(tag, "")

  return text